# BayMorph — Fast Simulated Single-Sérsic Fit

This notebook is the smallest complete BayMorph example. It creates one simple synthetic Sérsic galaxy, writes it as a FITS observation, generates the BayMorph configuration with the public config maker, and runs the same inference workflow used for real data.

Run the notebook from the repository root. The simulation is intentionally small and the sampler settings are deliberately modest so that the example remains suitable for interactive use.

In [ ]:
from pathlib import Path

import numpy as np

from baymorph import BayMorphRunner, SyntheticSersic, make_config_from_fits
from baymorph.plotting import plot_log_image
from baymorph.simulation import make_observation

## 1. Create a simple synthetic observation

The simulation truth is explicit and independent of the inference configuration. The same production Sérsic forward model is used to generate the synthetic galaxy.

In [ ]:
work_dir = Path("runs/tutorial_simulated_single_sersic")
fits_path = work_dir / "input" / "simulated_single_sersic.fits"
config_path = work_dir / "simulated_single_sersic.yaml"

shape = (81, 81)
center = (40.0, 40.0)
psf_fwhm = 2.0

truth = {
    "flux": 20_000.0,
    "re": 6.0,
    "n": 2.0,
    "q": 0.75,
    "theta": np.deg2rad(25.0),
    "x0": center[0],
    "y0": center[1],
}

galaxy = SyntheticSersic(**truth)
simulation = make_observation(
    galaxy=galaxy,
    shape=shape,
    sky=20.0,
    psf_fwhm=psf_fwhm,
    poisson_noise=False,
    read_noise_sigma=2.0,
    seed=42,
)
simulation.to_fits(fits_path)

In [ ]:
plot_log_image(
    simulation.image,
    title="Simulated single-Sérsic observation",
    colorbar_label="Counts",
    show=True,
)


## 2. Generate the BayMorph configuration

`make_config_from_fits()` is the canonical single-observation config maker for both simulated and observed FITS images. Scientific priors remain explicit user choices; they are not estimated from the image being fitted.

In [ ]:
config = make_config_from_fits(
    fits_path,
    target_position=center,
    output_path=config_path,
    object_name="tutorial_simulated_single_sersic",
    model_name="single_sersic",
    model_priors={
        "flux": {"type": "uniform", "lower": 5_000.0, "upper": 50_000.0},
        "re": {"type": "uniform", "lower": 2.0, "upper": 12.0},
        "n": {"type": "uniform", "lower": 0.8, "upper": 4.0},
        "q": {"type": "uniform", "lower": 0.5, "upper": 1.0},
        "theta": {"type": "uniform", "lower": "-pi/2", "upper": "pi/2"},
        "x0": {
            "type": "uniform",
            "lower": -3.0,
            "upper": 3.0,
            "relative_to": "target_x",
        },
        "y0": {
            "type": "uniform",
            "lower": -3.0,
            "upper": 3.0,
            "relative_to": "target_y",
        },
        "sky": {"type": "uniform", "lower": -10.0, "upper": 10.0},
    },
    likelihood={
        "type": "gaussian",
        "include_normalization": True,
    },
    sampler={
        "name": "nautilus",
        "init": {"n_live": 100, "seed": 42},
        "run": {"n_eff": 200, "f_live": 0.05},
    },
    outputs_root="runs",
    cutout_mode="fixed",
    fixed_size=81,
    psf_type="gaussian",
    gaussian_fwhm_pixels=psf_fwhm,
)

config

## 3. All-in-one workflow

For normal use, this is the recommended interface. BayMorph performs image preparation, fitting, persistence, and the configured diagnostics.

In [ ]:
runner = BayMorphRunner(config_path)
result = runner.run(diagnostics=True)
result

## 4. Step-by-step equivalent

The same analysis can be executed stage by stage when inspection or debugging is useful. Because this example uses an analytic Gaussian PSF, there is no empirical-PSF construction stage.

```python
runner = BayMorphRunner(config_path)
runner.prepare_image()
result = runner.fit()
runner.diagnostics()
```

For real data with an empirical PSF, the explicit workflow additionally uses `select_psf_candidates()` and `build_psf()` before `fit()`.